# FEVER Data Pipeline
# Step 0: Download all required files

In [ ]:
import os
import urllib.request

DATA_DIR = "/Users/zheng/CIS517-RL" # Change this to your desired data directory for reproduction

files_to_download = [
    ("train.jsonl",           "https://fever.ai/download/fever/train.jsonl"),
    ("shared_task_dev.jsonl", "https://fever.ai/download/fever/shared_task_dev.jsonl"),
    ("paper_test.jsonl",      "https://fever.ai/download/fever/paper_test.jsonl"),
    ("wiki-pages.zip",        "https://fever.ai/download/fever/wiki-pages.zip"),
]

for fname, url in files_to_download:
    dest = DATA_DIR + "/" + fname
    if os.path.exists(dest):
        print(fname, "already exists, skipping")
    else:
        print("downloading", fname, "... (this may take a while for wiki-pages.zip)")
        urllib.request.urlretrieve(url, dest)
        size_mb = os.path.getsize(dest) / 1e6
        print(f"done, {fname} ({size_mb:.1f} MB)")

train.jsonl already exists, skipping
shared_task_dev.jsonl already exists, skipping
paper_test.jsonl already exists, skipping
wiki-pages.zip already exists, skipping


## Step 1: Build the Wikipedia sentence lookup (run once, then pickle)

In [2]:
import json
import zipfile
import pickle

wiki_zip = DATA_DIR + "/wiki-pages.zip"
wiki_pkl = DATA_DIR + "/wiki_lookup.pkl"

if os.path.exists(wiki_pkl):
    print("wiki_lookup.pkl found, loading it...")
    with open(wiki_pkl, "rb") as f:
        wiki = pickle.load(f)
    print("loaded", len(wiki), "articles")
else:
    wiki = {}
    with zipfile.ZipFile(wiki_zip) as z:
        jsonl_files = [n for n in z.namelist() if n.endswith(".jsonl")]
        for i, fname in enumerate(jsonl_files):
            if i % 50 == 0:
                print(f"processing file {i}/{len(jsonl_files)}, {len(wiki)} articles so far")
            with z.open(fname) as f:
                for raw in f:
                    line = raw.decode("utf-8", errors="ignore").strip()
                    if not line:
                        continue
                    try:
                        page = json.loads(line)
                    except json.JSONDecodeError:
                        continue
                    sentences = {}
                    for sent_line in page["lines"].split("\n"):
                        if "\t" in sent_line:
                            parts = sent_line.split("\t")
                            try:
                                # parts[0] = sentence index, parts[1] = sentence text, rest = wiki links
                                sentences[int(parts[0])] = parts[1].strip()
                            except (ValueError, IndexError):
                                pass
                    wiki[page["id"]] = sentences

    with open(wiki_pkl, "wb") as f:
        pickle.dump(wiki, f)
    print("done! indexed", len(wiki), "articles, saved to wiki_lookup.pkl")

processing file 0/218, 0 articles so far
processing file 50/218, 1250000 articles so far
processing file 100/218, 2500000 articles so far
processing file 150/218, 3750000 articles so far
processing file 200/218, 5000000 articles so far
done! indexed 5416537 articles, saved to wiki_lookup.pkl


## Step 2: Join claims with passages

In [3]:
import random
from collections import defaultdict

random.seed(42)

label_map = {
    "SUPPORTS":        "SUPPORTED",
    "REFUTES":         "CONTRADICTED",
    "NOT ENOUGH INFO": "NOT MENTIONED",
}

def get_passage(evidence, wiki):
    first_set = evidence[0]
    parts = []
    for _, _, wiki_url, sent_id in first_set:
        if wiki_url and sent_id is not None:
            text = wiki.get(wiki_url, {}).get(int(sent_id), "")
            if text:
                parts.append(text)
    return " ".join(parts)

def build_examples(jsonl_path, wiki, target_per_class=None):
    buckets = defaultdict(list)
    used_articles = set()

    with open(jsonl_path) as f:
        for line in f:
            obj = json.loads(line)
            label = label_map.get(obj.get("label", ""))
            if not label:
                continue

            claim    = obj["claim"]
            evidence = obj.get("evidence", [])

            if label in ("SUPPORTED", "CONTRADICTED"):
                passage = get_passage(evidence, wiki)
                if not passage:
                    continue  # drop if wiki lookup failed
                for _, _, wiki_url, _ in evidence[0]:
                    if wiki_url:
                        used_articles.add(wiki_url)
            else:
                passage = None

            buckets[label].append({"claim": claim, "passage": passage, "label": label, "id": obj["id"]})

    # build distractor pool once, skip articles with no sentences
    distractor_pool = [a for a in wiki if a not in used_articles and wiki[a]]
    for ex in buckets["NOT MENTIONED"]:
        while True:
            article = random.choice(distractor_pool)
            sents = [s for s in wiki[article].values() if s]
            if sents:
                ex["passage"] = random.choice(sents)
                break

    if target_per_class:
        for lbl in buckets:
            random.shuffle(buckets[lbl])
            buckets[lbl] = buckets[lbl][:target_per_class]

    all_examples = []
    for lbl in buckets:
        all_examples.extend(buckets[lbl])
    random.shuffle(all_examples)
    return all_examples

## Step 3: Build train / dev / test joined files
### train: 5,000 total (~1,667 per class)
### dev:   full shared_task_dev (used for hyperparameter tuning)
### test:  full paper_test      (final evaluation, never seen during training)

In [4]:
# train: ~1667 per class = ~5k total, dev/test: take everything
splits = [
    ("train", DATA_DIR + "/train.jsonl",           1667),
    ("dev",   DATA_DIR + "/shared_task_dev.jsonl", None),
    ("test",  DATA_DIR + "/paper_test.jsonl",      None),
]

for name, path, tpc in splits:
    out_path = DATA_DIR + f"/fever_{name}_joined.jsonl"
    if os.path.exists(out_path):
        print(out_path, "already exists, skipping")
        continue

    print("building", name, "...")
    examples = build_examples(path, wiki, target_per_class=tpc)

    with open(out_path, "w") as f:
        for ex in examples:
            f.write(json.dumps(ex) + "\n")

    counts = defaultdict(int)
    for ex in examples:
        counts[ex["label"]] += 1
    print(name, "->", len(examples), "examples:", dict(counts))

building train ...
train -> 5001 examples: {'CONTRADICTED': 1667, 'NOT MENTIONED': 1667, 'SUPPORTED': 1667}
building dev ...
dev -> 19891 examples: {'CONTRADICTED': 6612, 'NOT MENTIONED': 6666, 'SUPPORTED': 6613}
building test ...
test -> 9946 examples: {'CONTRADICTED': 3304, 'NOT MENTIONED': 3333, 'SUPPORTED': 3309}
